<a href="https://colab.research.google.com/github/madinayergeshbay-sudo/2026_Spring_Application_Programming/blob/main/%EA%B3%BC%EC%A0%9C26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# [Step 1] 데이터 로드 및 샘플링
categories = ['comp.graphics', 'sci.space', 'talk.religion.misc']
newsgroups = fetch_20newsgroups(subset='train', categories=categories,
                                remove=('headers', 'footers', 'quotes'))

data, labels = [], []

for i, category in enumerate(categories):
    indices = np.where(newsgroups.target == i)[0][:20]  # 각 주제별 20개 추출
    for idx in indices:
        data.append(newsgroups.data[idx])
        labels.append(newsgroups.target_names[i])

# [Step 2] CountVectorizer 설정 및 변환
# TODO: CountVectorizer를 선언하고 데이터를 fit_transform 하세요.
vectorizer = CountVectorizer(stop_words='english')
count_matrix = vectorizer.fit_transform(data)

# [Step 3] 분류 함수 구현
def classify_text(input_text):
    # TODO: 입력 텍스트를 벡터화하고 코사인 유사도를 계산하는 로직을 작성하세요.
    input_vec = vectorizer.transform([input_text])
    sim = cosine_similarity(input_vec, count_matrix)

    best_idx = np.argmax(sim)
    return labels[best_idx], sim[0][best_idx]

# [Step 4] 테스트 실행
test_sentences = [
    "The rocket launched into orbit.",
    "A new 3D rendering technique for graphics.",
    "Theological discussions on faith and god."
]

for s in test_sentences:
    cat, score = classify_text(s)
    print(f"문장: {s[:30]}... | 예측: {cat} | 유사도: {score:.4f}")

문장: The rocket launched into orbit... | 예측: sci.space | 유사도: 0.0870
문장: A new 3D rendering technique f... | 예측: comp.graphics | 유사도: 0.1952
문장: Theological discussions on fai... | 예측: talk.religion.misc | 유사도: 0.3430


CountVectorizer는 학습 데이터에 등장한 단어만 사전에 저장하여 벡터를 생성한다.
따라서 입력 문장에 포함된 단어가 학습 데이터의 단어 사전에 없으면 입력 벡터가 거의 0이 된다.
이 경우 기존 문서들과 공통 단어가 없기 때문에 cosine similarity 값이 0.0000으로 나올 수 있다.
또한 각 주제별 샘플 수를 20개만 사용하면 단어 사전의 크기가 작아져 새로운 문장을 충분히 반영하지 못할 수 있다.

각 주제별 샘플 수를 20개에서 100개로 늘리면 더 다양한 단어와 표현이 포함된다.
그래서 입력 문장과 겹치는 단어가 많아질 가능성이 커지고 유사도 점수와 분류 성능이 향상될 수 있다.

CountVectorizer는 단어 빈도만 사용하지만 TfidfVectorizer는 중요한 단어에 더 높은 가중치를 준다.
따라서 주제를 잘 나타내는 단어가 강조되어 분류 정확도가 좋아질 수 있다.

CountVectorizer에서 ngram_range=(1, 2)를 사용하면 한 단어뿐만 아니라 두 단어 묶음도 특징으로 사용할 수 있다.
예를 들어 "space shuttle", "3D rendering"과 같은 표현을 하나의 특징으로 처리하여 문맥 정보를 더 잘 반영할 수 있다.
따라서 분류 성능이 향상될 수 있다.

In [3]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import gradio as gr

categories = ['comp.graphics', 'sci.space', 'talk.religion.misc']
newsgroups = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes')
)

data, labels = [], []

for i, category in enumerate(categories):
    indices = np.where(newsgroups.target == i)[0][:20]
    for idx in indices:
        data.append(newsgroups.data[idx])
        labels.append(newsgroups.target_names[i])

vectorizer = CountVectorizer(stop_words='english')
count_matrix = vectorizer.fit_transform(data)

def classify_text(input_text):
    if input_text.strip() == "":
        return "텍스트를 입력하세요.", "0.0000"

    input_vec = vectorizer.transform([input_text])
    sim = cosine_similarity(input_vec, count_matrix)

    best_idx = np.argmax(sim)
    return labels[best_idx], f"{sim[0][best_idx]:.4f}"

demo = gr.Interface(
    fn=classify_text,
    inputs=gr.Textbox(lines=5, label="텍스트 입력"),
    outputs=[
        gr.Textbox(label="예측 주제"),
        gr.Textbox(label="유사도")
    ],
    title="뉴스그룹 주제 분류 서비스",
    description="텍스트를 입력하면 어떤 주제인지 구분합니다."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5b3f420906864ccdd8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
